In [1]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [2]:
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Make some data
N = 1
T = 10
D = 3
K = 2
X = np.random.randn(N, T, D)

In [4]:
# Make an RNN
M = 5 # number of hidden units
i = Input(shape=(T, D))
x = SimpleRNN(M)(i)
x = Dense(K)(x)

model = Model(i, x)

In [5]:
# Get the output
Yhat = model.predict(X)
print(Yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
[[0.10973966 1.0237576 ]]


In [6]:
# See if we can replicate this output
# Get the weights first
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57 (228.00 B)

 Trainable params: 57 (228.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
# See what's returned
model.layers[1].get_weights()

[array([[-0.29807514,  0.05722022,  0.65307087,  0.25444287, -0.27938443],
        [-0.6730536 , -0.33543044, -0.43353963,  0.8411483 , -0.0711264 ],
        [ 0.6181585 , -0.10248238,  0.581888  , -0.45537472,  0.8526116 ]],
       dtype=float32),
 array([[ 0.48153925, -0.3023201 ,  0.6967198 , -0.33227408, -0.28442553],
        [ 0.07908003, -0.8905695 , -0.42216012,  0.11826857, -0.09179093],
        [ 0.16324896,  0.26465896, -0.29299188,  0.14530866, -0.8923826 ],
        [-0.81783295, -0.21309866,  0.40399632,  0.13295016, -0.3238047 ],
        [ 0.25761467, -0.00574181,  0.29547444,  0.9147765 ,  0.09736751]],
       dtype=float32),
 array([0., 0., 0., 0., 0.], dtype=float32)]

In [8]:
# Check their shapes
# Should make sense
# First output is input > hidden
# Second output is hidden > hidden
# Third output is bias term (vector of length M)
a, b, c = model.layers[1].get_weights()
print(a.shape, b.shape, c.shape)

(3, 5) (5, 5) (5,)


In [9]:
Wx, Wh, bh = model.layers[1].get_weights()
Wo, bo = model.layers[2].get_weights()
h_last = np.zeros(M) # initial hidden state
x = X[0] # the one and only sample
Yhats = [] # where we store the outputs

for t in range(T):
  h = np.tanh(x[t].dot(Wx) + h_last.dot(Wh) + bh)
  y = h.dot(Wo) + bo # we only care about this value on the last iteration
  Yhats.append(y)

  # important: assign h to h_last
  h_last = h

# print the final output
print(Yhats[-1])

[0.10973966 1.02375767]
